# Proteome of JCVI-syn1.0 and JCVI-syn3A

Authors: Troy Brier, Enguang Fu

Absolute protein quantification, functional annotation and cross-organism
comparison for both synthetic cells, from mass-spectrometry iBAQ to a distributable
proteome table.

## Background

Mass spectrometry reports **iBAQ** (intensity-based absolute quantification), a
proxy for protein copy number that does not necessarily scale linearly with the
true count. This notebook converts iBAQ into copies per cell in two steps.

1. **iBAQ → iPM** (iBAQ per million), per replicate:

   $$\mathrm{iPM}_i = \frac{\mathrm{iBAQ}_i}{\sum_j \mathrm{iBAQ}_j}\times 10^6$$

   Each replicate is normalised to its own total, so replicates become comparable
   regardless of loading. This mirrors TPM for RNA.

2. **iPM → copies per cell**, by mass balance. The iPM-weighted mean protein mass
   gives the average mass of one protein molecule; the cell's total protein mass
   divided by that gives the number of molecules:

   $$\bar{M} = \sum_i M_i\,\frac{\mathrm{iPM}_i}{\sum_j \mathrm{iPM}_j}
   \qquad
   N_\mathrm{total} = \frac{\mathrm{gDW}\times f_\mathrm{protein}}{\bar{M}/N_A}
   \qquad
   n_i = \frac{\mathrm{iPM}_i}{10^6}\,N_\mathrm{total}$$

   Molecular weights come from the GenBank CDS translations for syn1, and from the
   protein FASTA for syn3A.

**Localization** (syn1) follows the scheme of *Assembly of Macromolecular Complexes
in the Whole-cell Model of a Minimal Cell*
([doi:10.1021/acs.jpcb.5c04532](https://doi.org/10.1021/acs.jpcb.5c04532)):
SignalP 6 (secreted / lipoprotein) takes priority over DeepTMHMM (membrane),
otherwise cytoplasmic. It exists so membrane proteins can be excluded when
correlating transcriptome against proteome.

**Functional annotation** (syn3A) is a hand-curated Primary > Secondary > Tertiary
hierarchy. `Syn3A_annotation/annotate_tertiary_function_syn3A.py` drafted it from
the metabolic reconstruction and a controlled vocabulary; a human then adjudicated
every CONFLICT / PRIMARY_MISMATCH / AI row. **That curated workbook is a primary
input, not a pipeline output** — re-running the draft script does not reproduce it.
The script is kept beside the workbook for provenance only.

## Inputs

| | |
|---|---|
| `Processed_proteomics_canonical/` | Spectronaut iBAQ, both organisms (MS-derived, not regenerable here) |
| `Syn3A_annotation/` | curated function hierarchy + controlled vocabulary |
| `DeepTMHMM_Topology/`, `SignalP_prediction/` | third-party topology predictions over our own sequences |
| `../Genomes_Input/` | syn1 GFF3 + GenBank, syn3A protein FASTA |
| `../Syn1_Syn3A_Transcriptomics/` | syn3A absolute mRNA copies per cell and Illumina TPM |

## Outputs

| | |
|---|---|
| `syn1_proteome.tsv`, `syn3A_proteome.tsv` | machine-readable, read by every downstream stage |
| `syn1_proteome_2026.xlsx`, `Syn3A_proteome_2026.xlsx` | for distribution, with a column legend sheet |
| `syn3A_proteome.html` | interactive: click a function, get its proteins |
| `syn1_vs_syn3A_proteome.html` | interactive syn1 vs syn3A comparison |
| `Proteome.txt` | run log |

## Summary of outcome

Filled in from the run — see the final cell.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

import proteome_common as pc

# ── Inputs ───────────────────────────────────────────────────────────────────
ROOT          = Path("..")
SYN1_GFF      = ROOT / "Genomes_Input/syn1.genes.gff3"
SYN1_GB       = ROOT / "Genomes_Input/syn1.gb"
SYN1_FAA      = ROOT / "Genomes_Input/syn1_proteins.faa"
SYN3A_FAA     = ROOT / "Genomes_Input/syn3A_ptns.fasta"
SYN3A_GB      = ROOT / "Genomes_Input/syn3a.gb"
SYN3A_RNA     = ROOT / "Syn1_Syn3A_Transcriptomics/syn3A_rna_abundances.tsv"
SYN1_MS       = Path("Processed_proteomics_canonical/Syn1.0_newSearch_20260402.xlsx")
SYN3A_MS      = Path("Processed_proteomics_canonical/Syn3_summary.xlsx")
DEEPTMHMM     = Path("DeepTMHMM_Topology/TMRs_syn1.gff3")
SIGNALP       = Path("SignalP_prediction/output.gff3")
SYN3A_CURATED = Path("Syn3A_annotation/syn3A_proteome_fully_annotated_Revised.xlsx")
HIER          = Path("Syn3A_annotation/function_hierachy.tsv")

# ── Outputs ──────────────────────────────────────────────────────────────────
OUT_SYN1_TSV   = Path("syn1_proteome.tsv")
OUT_SYN3A_TSV  = Path("syn3A_proteome.tsv")
OUT_SYN1_XLSX  = Path("syn1_proteome_2026.xlsx")
OUT_SYN3A_XLSX = Path("Syn3A_proteome_2026.xlsx")
OUT_SYN3A_HTML = Path("syn3A_proteome.html")
OUT_CMP_HTML   = Path("syn1_vs_syn3A_proteome.html")
OUT_LOG        = Path("Proteome.txt")

# every printed line is mirrored into Proteome.txt (OUTPUT.md convention)
_LOG = []
def say(line=""):
    print(line)
    _LOG.append(str(line))

def rule(title):
    say("")
    say("=" * 74)
    say(title)
    say("=" * 74)

missing = [str(p) for p in [SYN1_GFF, SYN1_GB, SYN1_FAA, SYN3A_FAA, SYN3A_GB, SYN3A_RNA, SYN1_MS,
                            SYN3A_MS, DEEPTMHMM, SIGNALP, SYN3A_CURATED, HIER]
           if not p.exists()]
assert not missing, "missing inputs:\n  " + "\n  ".join(missing)
rule("Proteome of JCVI-syn1.0 and JCVI-syn3A")
say("all 12 inputs present")


Proteome of JCVI-syn1.0 and JCVI-syn3A
all 12 inputs present


---
# Part A — syn1 proteome

911 annotated syn1 loci, of which the mass spectrometry detects a subset. Every
locus is kept (including RNA genes and pseudogenes) so downstream joins never
silently drop a row; undetected loci simply carry blank iPM.

**Gene annotation** — one row per `gene` feature of the syn1 GFF3.

In [2]:
def parse_attrs(attr):
    """GFF3 attribute string -> dict."""
    out = {}
    for item in attr.split(";"):
        item = item.strip()
        if "=" in item:
            k, v = item.split("=", 1)
            out[k] = v
    return out

rows = []
with open(SYN1_GFF) as fh:
    for line in fh:
        if not line.strip() or line.startswith("#"):
            continue
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 9 or parts[2] != "gene":
            continue
        chrom, _src, _feat, s1, e1, _score, strand, _phase, attrs = parts
        a = parse_attrs(attrs)
        rows.append((chrom, int(s1) - 1, int(e1), strand,       # GFF3 is 1-based inclusive
                     a.get("locus_tag", ""), a.get("gene", ""),
                     a.get("rna_type", ""), a.get("product", "")))

syn1 = (pd.DataFrame(rows, columns=["chrom", "start0", "end0", "strand", "locus_tag",
                                    "gene_name", "rna_type", "gene_product"])
          .sort_values(["chrom", "start0", "end0"]).reset_index(drop=True))

rule("Part A - syn1 proteome")
say(f"loci parsed from {SYN1_GFF.name}: {len(syn1)}")
for k, v in syn1["rna_type"].value_counts(dropna=False).items():
    say(f"  {str(k) or '(blank)':<10} {v:>4}")
syn1.head()


Part A - syn1 proteome
loci parsed from syn1.genes.gff3: 911
  mRNA        828
  pseudo       42
  tRNA         30
  rRNA          6
  ncRNA         4
  tmRNA         1


,chrom,start0,end0,strand,locus_tag,gene_name,rna_type,gene_product
0,CP002027.1,0,1353,+,MMSYN1_0001,dnaA_1,mRNA,chromosomal replication initiator protein DnaA
1,CP002027.1,1510,2638,+,MMSYN1_0002,dnaN,mRNA,DNA polymerase III_ beta subunit
2,CP002027.1,2674,3217,+,MMSYN1_0003,rnmV,mRNA,ribonuclease M5
3,CP002027.1,3206,4007,+,MMSYN1_0004,ksgA,mRNA,dimethyladenosine transferase
4,CP002027.1,4063,5155,+,MMSYN1_0005,,mRNA,hypothetical purine NTPase


**Molecular weight** — from every CDS translation in the syn1 GenBank record.

In [3]:
mw_map = pc.mw_from_genbank(str(SYN1_GB))
syn1["protein_mw_Da"] = syn1["locus_tag"].map(mw_map)
say(f"\nprotein MW from {SYN1_GB.name}: {syn1['protein_mw_Da'].notna().sum()} / {len(syn1)} loci")


protein MW from syn1.gb: 828 / 911 loci


**iBAQ → iPM.** Spectronaut reports three replicates. Protein groups whose
accessions are `;`-separated share one iBAQ value, so the first field is taken.

In [4]:
prot = pd.read_excel(SYN1_MS, sheet_name="Proteins_all")
prot["locus_tag"] = prot["PG.ProteinAccessions"]
prot = prot[prot["locus_tag"].str.startswith("MMSYN1_")].copy()

ibaq_cols = [c for c in prot.columns if "IBAQ" in c]
prot = prot[["locus_tag"] + ibaq_cols].copy()
for c in ibaq_cols:                      # ;-separated groups carry identical values
    prot[c] = pd.to_numeric(prot[c].astype(str).str.split(";").str[0], errors="coerce")

prot, REPS1 = pc.ibaq_to_ipm(prot, ibaq_cols)

say(f"\nsyn1 proteins in {SYN1_MS.name}: {len(prot)}  ({len(ibaq_cols)} replicates)")
say("  iBAQ NaNs per replicate: "
    + ", ".join(f"{c.split('.')[-1]}={int(prot[c].isna().sum())}" for c in ibaq_cols))
say(f"  median CV across replicates: {prot['iPM_CV'].median():.3f}")

ipm = prot.set_index("locus_tag")[REPS1 + ["iPM_mean", "iPM_CV"]]
for c in ipm.columns:
    syn1[c] = syn1["locus_tag"].map(ipm[c])

n_det = int(syn1["iPM_mean"].notna().sum())
n_mrna = int((syn1["rna_type"] == "mRNA").sum())
say(f"  detected: {n_det} loci ({100*n_det/n_mrna:.1f}% of {n_mrna} mRNA genes)")


syn1 proteins in Syn1.0_newSearch_20260402.xlsx: 722  (3 replicates)
  iBAQ NaNs per replicate: IBAQ=1, IBAQ=2, IBAQ=2
  median CV across replicates: 0.078
  detected: 721 loci (87.1% of 828 mRNA genes)


**Absolute copy number.** Cell dry mass and protein mass fraction set the total
protein budget; each protein takes its iPM share of it.

In [5]:
SYN1_gDW           = 1.2844977017695941e-14   # g, cell dry mass
SYN1_PTN_MASS_FRAC = 0.582                    # 50.7-62.4% across measurements (Razin et al. 1963)
SYN1_RADIUS_NM     = 439 / 2                  # Roger-Reischer et al. Nature 2023

_idx = syn1.set_index("locus_tag")
avg_mw1, total1 = pc.total_proteins_per_cell(
    _idx["protein_mw_Da"], _idx, REPS1, SYN1_gDW, SYN1_PTN_MASS_FRAC)
vol1 = pc.sphere_volume_um3(SYN1_RADIUS_NM)

say(f"\nsyn1 cell: radius {SYN1_RADIUS_NM:.1f} nm -> volume {vol1:.3e} um^3")
say(f"           dry mass {SYN1_gDW:.4e} g, protein fraction {SYN1_PTN_MASS_FRAC}")
for i in range(len(REPS1)):
    say(f"  rep{i+1}: avg protein MW {avg_mw1[i]:>10,.1f} g/mol | "
        f"{total1[i]:>10,.0f} molecules/cell | {total1[i]/vol1:.3e} per um^3")
say(f"  mean total proteins/cell: {total1.mean():,.0f} +/- {total1.std():,.0f}")

syn1["ptn_copy_number"] = syn1["iPM_mean"] / 1e6 * total1.mean()
say(f"  copy number assigned to {int(syn1['ptn_copy_number'].notna().sum())} loci")

syn1.loc[syn1["ptn_copy_number"].notna(),
         ["locus_tag", "gene_name", "gene_product", "iPM_mean", "ptn_copy_number"]] \
    .sort_values("ptn_copy_number", ascending=False).head(10)


syn1 cell: radius 219.5 nm -> volume 4.430e-02 um^3
           dry mass 1.2845e-14 g, protein fraction 0.582
  rep1: avg protein MW   35,216.4 g/mol |    127,839 molecules/cell | 2.886e+06 per um^3
  rep2: avg protein MW   35,466.2 g/mol |    126,938 molecules/cell | 2.865e+06 per um^3
  rep3: avg protein MW   34,963.3 g/mol |    128,764 molecules/cell | 2.907e+06 per um^3
  mean total proteins/cell: 127,847 +/- 745
  copy number assigned to 721 loci


,locus_tag,gene_name,gene_product,iPM_mean,ptn_copy_number
155,MMSYN1_0151,tuf,translation elongation factor Tu,56105.410379,7172.911583
473,MMSYN1_0475,ldh,L-lactate dehydrogenase,36118.364805,4617.626634
229,MMSYN1_0225,pdhA,pyruvate dehydrogenase (acetyl-transferring) E...,26297.927956,3362.112687
227,MMSYN1_0223,,NADH oxidase (noxase),24828.045660,3174.192562
607,MMSYN1_0607,gap,glyceraldehyde-3-phosphate dehydrogenase_ type I,22384.962993,2861.851633
667,MMSYN1_0667,rpsS,ribosomal protein S19,19287.071426,2465.795314
665,MMSYN1_0665,rpsC,ribosomal protein S3,17799.490460,2275.612466
805,MMSYN1_0806,rplL,ribosomal protein L7/L12,17435.544237,2229.083013
64,MMSYN1_0059,ald,alanine dehydrogenase,15177.333843,1940.377461
658,MMSYN1_0658,rpsN,ribosomal protein S14p/S29e,15017.568664,1919.951953


**Localization.** DeepTMHMM gives transmembrane regions, SignalP 6 gives secretion
signals. Priority: SignalP (extracellular / lipoprotein) > DeepTMHMM (membrane) >
cytoplasmic. Non-coding genes are marked separately rather than guessed at.

In [6]:
# ── DeepTMHMM: "# <locus_tag ...> Number of predicted TMRs: N" then region lines ──
LOCUS_RE = re.compile(r"(MMSYN1_\d+)")
topo, cur = {}, None
with open(DEEPTMHMM) as fh:
    for line in fh:
        line = line.rstrip("\n")
        if not line or line.startswith("##") or line == "//":
            continue
        if line.startswith("#"):
            m = LOCUS_RE.search(line[2:].strip())
            if not m:
                continue
            cur = m.group(1)
            topo.setdefault(cur, {"TMRs": 0, "length": None, "regions": []})
            if "Length:" in line:
                topo[cur]["length"] = int(re.search(r"Length:\s*(\d+)", line).group(1))
            elif "Number of predicted TMRs:" in line:
                topo[cur]["TMRs"] = int(re.search(r"TMRs:\s*(\d+)", line).group(1))
        else:
            # region line: <seq_id> <inside|TMhelix|outside|signal> <start> <end>
            f = line.split("\t")
            if len(f) >= 4 and cur is not None:
                topo[cur]["regions"].append((f[1].strip(), int(f[2]), int(f[3])))

say(f"\nDeepTMHMM proteins parsed: {len(topo)}")
say(f"  transmembrane (TMRs > 0): {sum(1 for v in topo.values() if v['TMRs'] > 0)}")
say(f"  no TMR         (TMRs = 0): {sum(1 for v in topo.values() if v['TMRs'] == 0)}")
say(f"  topology segments: {sum(len(v['regions']) for v in topo.values())} across "
    + ", ".join(f"{k} {sum(1 for v in topo.values() for r in v['regions'] if r[0] == k)}"
                for k in ("signal", "inside", "TMhelix", "outside")))

# ── SignalP 6 ────────────────────────────────────────────────────────────────
signalp = {}
with open(SIGNALP) as fh:
    for line in fh:
        if not line.strip() or line.startswith("#"):
            continue
        parts = line.rstrip("\n").split("\t")
        if len(parts) < 3:
            continue
        locus_tag = parts[0].split("_gene_")[0]
        if parts[2].strip() == "signal_peptide":
            signalp[locus_tag] = "extracellular"
        elif parts[2].strip() == "lipoprotein_signal_peptide":
            signalp[locus_tag] = "lipoprotein"

say(f"SignalP 6 predictions: {len(signalp)}"
    f"  (extracellular {sum(1 for v in signalp.values() if v == 'extracellular')},"
    f" lipoprotein {sum(1 for v in signalp.values() if v == 'lipoprotein')})")

syn1["TMRs"]           = syn1["locus_tag"].map(lambda t: topo.get(t, {}).get("TMRs"))
syn1["protein_length"] = syn1["locus_tag"].map(lambda t: topo.get(t, {}).get("length"))
syn1["signalP"]        = syn1["locus_tag"].map(signalp)

def localize(row):
    if row["rna_type"] != "mRNA":
        return "Non_proteins"          # RNA gene: no prediction applies
    if pd.notna(row["signalP"]):
        return row["signalP"]
    if pd.notna(row["TMRs"]):
        return "membrane" if row["TMRs"] > 0 else "cytoplasmic"
    return "Non_predicted"

syn1["ptn_localization"] = syn1.apply(localize, axis=1)

say("\nsyn1 localization (mRNA genes):")
for k, v in syn1.loc[syn1["rna_type"] == "mRNA", "ptn_localization"].value_counts().items():
    say(f"  {k:<16} {v:>4}  ({100*v/n_mrna:.1f}%)")


DeepTMHMM proteins parsed: 828
  transmembrane (TMRs > 0): 162
  no TMR         (TMRs = 0): 666
  topology segments: 2799 across signal 125, inside 1124, TMhelix 923, outside 627
SignalP 6 predictions: 93  (extracellular 15, lipoprotein 78)

syn1 localization (mRNA genes):
  cytoplasmic       582  (70.3%)
  membrane          153  (18.5%)
  lipoprotein        78  (9.4%)
  extracellular      15  (1.8%)


**Write** the syn1 table.

In [7]:
SYN1_COLS = ["locus_tag", "gene_name", "gene_product", "rna_type",
             "chrom", "start0", "end0", "strand",
             "protein_mw_Da", "protein_length",
             "iPM_rep1", "iPM_rep2", "iPM_rep3", "iPM_mean", "iPM_CV",
             "ptn_copy_number", "TMRs", "signalP", "ptn_localization"]

# TSV = pipeline contract (keeps iPM, full precision);
# workbook = published view (no iPM, whole molecules).
SYN1_PUB_COLS = [c for c in SYN1_COLS if not c.startswith("iPM_")]

syn1_out = syn1[SYN1_COLS].copy()
syn1_out.to_csv(OUT_SYN1_TSV, sep="\t", index=False)
say(f"\nwrote {OUT_SYN1_TSV}  ({len(syn1_out)} loci, {len(SYN1_COLS)} columns)")
syn1_out.head()


wrote syn1_proteome.tsv  (911 loci, 19 columns)


,locus_tag,gene_name,gene_product,rna_type,chrom,start0,end0,strand,protein_mw_Da,protein_length,iPM_rep1,iPM_rep2,iPM_rep3,iPM_mean,iPM_CV,ptn_copy_number,TMRs,signalP,ptn_localization
0,MMSYN1_0001,dnaA_1,chromosomal replication initiator protein DnaA,mRNA,CP002027.1,0,1353,+,51702.7783,450.0,230.351328,222.215350,226.517840,226.361506,0.017981,28.939652,0.0,NaN,cytoplasmic
1,MMSYN1_0002,dnaN,DNA polymerase III_ beta subunit,mRNA,CP002027.1,1510,2638,+,42649.6889,375.0,1665.397226,1648.979491,1540.986670,1618.454462,0.041762,206.914639,0.0,NaN,cytoplasmic
2,MMSYN1_0003,rnmV,ribonuclease M5,mRNA,CP002027.1,2674,3217,+,20634.5673,180.0,303.351424,306.917822,291.795306,300.688184,0.026290,38.442099,0.0,NaN,cytoplasmic
3,MMSYN1_0004,ksgA,dimethyladenosine transferase,mRNA,CP002027.1,3206,4007,+,31176.8505,266.0,210.266633,234.845964,234.069194,226.393931,0.061716,28.943798,0.0,NaN,cytoplasmic
4,MMSYN1_0005,,hypothetical purine NTPase,mRNA,CP002027.1,4063,5155,+,42663.1420,363.0,433.667669,401.467029,396.125822,410.420173,0.049484,52.471011,4.0,NaN,membrane


---
# Part B — syn3A proteome

455 syn3A proteins with a curated Primary > Secondary > Tertiary function
hierarchy. This part joins four things onto that curated backbone: iBAQ-derived
abundance, protein sequences, absolute mRNA copies per cell, and Illumina TPM.

**Curated backbone, sequences and molecular weight.** Sequences come from syn3A's own
protein FASTA, and molecular weights are computed from them.

In [8]:
syn3a = pd.read_excel(SYN3A_CURATED, sheet_name=0)
syn3a = syn3a.rename(columns={
    "Locus Tag": "locus_tag", "Gene Name": "gene_name", "Gene Product": "gene_product",
    "Protein Length": "protein_length", "Exp. Ptn Cnt": "exp_ptn_cnt_2019",
    "Essentiality": "essentiality", "Localization": "localization",
    "Primary Function": "primary_function", "Secondary Function": "secondary_function",
    "Tertiary Function": "tertiary_function", "Sim. Initial Ptn Cnt": "sim_initial_ptn_cnt",
    "Review Flag": "review_flag", "Mismatch Solved": "mismatch_solved"})
syn3a["exp_ptn_cnt_2019"] = pd.to_numeric(syn3a["exp_ptn_cnt_2019"], errors="coerce")

def locus_num(s):
    return s.astype(str).str.extract(r"_(\d+)$")[0]

rule("Part B - syn3A proteome")
say(f"curated proteins in {SYN3A_CURATED.name}: {len(syn3a)}")

# ── the three pseudogenes ────────────────────────────────────────────────────
# The curated backbone comes from the complex-formation paper's protein table,
# which covers protein-coding genes only. syn3A's three pseudogenes ARE detected
# by mass spectrometry -- 0602 is among the most abundant entries in the sample --
# so they belong in the proteome table even though nobody has curated a function
# for them. NCBI gives them no translation, and naively translating the genomic
# span yields 1, 6 and 14 internal stop codons respectively, so protein sequence,
# length and molecular weight are left BLANK rather than fabricated. Carrying no
# MW, they stay out of the mass-balance average, and every other protein's copy
# number is therefore unchanged by their inclusion.
PSEUDO_LABEL = "Pseudogene"

_rec = next(__import__("Bio.SeqIO", fromlist=["SeqIO"]).parse(str(SYN3A_GB), "genbank"))
_inference = {f.qualifiers["locus_tag"][0]: f.qualifiers.get("inference", [""])[0].split(":")[-1]
              for f in _rec.features
              if f.type == "CDS" and "locus_tag" in f.qualifiers}
pseudo_rows = []
for f in _rec.features:
    if f.type != "gene" or "pseudo" not in f.qualifiers:
        continue
    lt = f.qualifiers["locus_tag"][0]
    ref = _inference.get(lt, "")
    pseudo_rows.append({
        "locus_tag": lt,
        "gene_product": "pseudogene" + (f", similar to {ref}" if ref else ""),
        "primary_function": PSEUDO_LABEL, "secondary_function": PSEUDO_LABEL,
        "tertiary_function": PSEUDO_LABEL, "review_flag": "PSEUDOGENE_NOT_CURATED"})

syn3a["is_pseudogene"] = False
syn3a = pd.concat([syn3a, pd.DataFrame(pseudo_rows).assign(is_pseudogene=True)],
                  ignore_index=True).sort_values("locus_tag").reset_index(drop=True)
say(f"  + {len(pseudo_rows)} pseudogenes from {SYN3A_GB.name}: "
    + ", ".join(r["locus_tag"] for r in pseudo_rows))
say(f"  syn3A proteome rows: {len(syn3a)}")

syn3a["locus_num"] = locus_num(syn3a["locus_tag"])
syn1["locus_num"]  = locus_num(syn1["locus_tag"])

# Sequence and molecular weight come from syn3A's OWN protein FASTA, not from the
# syn1 ortholog by locus number. Inheriting from syn1 leaves 13 detected proteins
# with no MW -- their syn1 counterpart is itself a pseudogene, or does not exist
# (the syn3A-only loci) -- and it is simply wrong wherever the two orthologs
# differ: JCVISYN3A_0592 is 399 aa here, but the syn1 MW implies ~530 aa. The
# FASTA length matches the curated Protein Length for all 445 proteins carrying
# both, so the FASTA is the authority. Pseudogenes have no entry and stay blank.
seqs = pc.read_fasta(str(SYN3A_FAA))
syn3a["protein_sequence"] = syn3a["locus_tag"].map(lambda t: seqs.get(t, "").rstrip("*"))
syn3a["protein_mw_Da"] = syn3a["protein_sequence"].map(pc.mw_from_sequence)

_nseq = int((syn3a["protein_sequence"] != "").sum())
say(f"  protein sequences from {SYN3A_FAA.name}: {_nseq} / {len(syn3a)}")
say(f"  MW computed from them: {int(syn3a['protein_mw_Da'].notna().sum())} / {len(syn3a)}"
    f"  (blank for the {int(syn3a['is_pseudogene'].sum())} pseudogenes -- no product)")


Part B - syn3A proteome
curated proteins in syn3A_proteome_fully_annotated_Revised.xlsx: 455
  + 3 pseudogenes from syn3a.gb: JCVISYN3A_0051, JCVISYN3A_0546, JCVISYN3A_0602
  syn3A proteome rows: 458
  protein sequences from syn3A_ptns.fasta: 455 / 458
  MW computed from them: 455 / 458  (blank for the 3 pseudogenes -- no product)


**iBAQ → iPM and absolute copy number**, same two steps as syn1 with syn3A constants.

In [9]:
syn3_raw = pd.read_excel(SYN3A_MS, sheet_name="Proteins_all_raw data")
syn3_raw["locus_tag"] = syn3_raw["PG.ProteinDescriptions"].str.extract(r"\[locus_tag=(\S+?)\]")
ibaq3 = [c for c in syn3_raw.columns if "IBAQ" in c]
s3 = syn3_raw[["locus_tag"] + ibaq3].copy()
for c in ibaq3:
    s3[c] = pd.to_numeric(s3[c].astype(str).str.split(";").str[0], errors="coerce")

s3, REPS3 = pc.ibaq_to_ipm(s3, ibaq3)
say(f"\nsyn3A proteins in {SYN3A_MS.name}: {len(s3)}  ({len(ibaq3)} replicates)")
say(f"  median CV across replicates: {s3['iPM_CV'].median():.3f}")

ipm3 = s3.set_index("locus_tag")[REPS3 + ["iPM_mean", "iPM_CV"]]
for c in ipm3.columns:
    syn3a[c] = syn3a["locus_tag"].map(ipm3[c])
n_det3 = int(syn3a["iPM_mean"].notna().sum())
say(f"  detected: {n_det3} / {len(syn3a)} ({100*n_det3/len(syn3a):.1f}%)")

# Syn3A physical constants (Breuer et al. eLife 2019)
SYN3A_gDW           = 1.0161e-14
SYN3A_PTN_MASS_FRAC = 54.727 / 100
SYN3A_RADIUS_NM     = 200

_idx3 = syn3a.set_index("locus_tag")
avg_mw3, total3 = pc.total_proteins_per_cell(
    _idx3["protein_mw_Da"], _idx3, REPS3, SYN3A_gDW, SYN3A_PTN_MASS_FRAC)
vol3 = pc.sphere_volume_um3(SYN3A_RADIUS_NM)

say(f"\nsyn3A cell: radius {SYN3A_RADIUS_NM} nm -> volume {vol3:.3e} um^3")
say(f"            dry mass {SYN3A_gDW:.4e} g, protein fraction {SYN3A_PTN_MASS_FRAC:.5f}")
for i in range(len(REPS3)):
    say(f"  rep{i+1}: avg protein MW {avg_mw3[i]:>10,.1f} g/mol | "
        f"{total3[i]:>10,.0f} molecules/cell | {total3[i]/vol3:.3e} per um^3")
say(f"  mean total proteins/cell: {total3.mean():,.0f} +/- {total3.std():,.0f}")

syn3a["exp_ptn_cnt_2026"] = syn3a["iPM_mean"] / 1e6 * total3.mean()

_ps = syn3a["is_pseudogene"]
say(f"  pseudogenes carry {100 * syn3a.loc[_ps, 'iPM_mean'].sum() / 1e6:.2f}% of the iPM pool "
    "but have no MW, so they do not enter the average above")


syn3A proteins in Syn3_summary.xlsx: 449  (3 replicates)
  median CV across replicates: 0.046
  detected: 449 / 458 (98.0%)

syn3A cell: radius 200 nm -> volume 3.351e-02 um^3
            dry mass 1.0161e-14 g, protein fraction 0.54727
  rep1: avg protein MW   33,398.8 g/mol |    100,267 molecules/cell | 2.992e+06 per um^3
  rep2: avg protein MW   33,427.2 g/mol |    100,182 molecules/cell | 2.990e+06 per um^3
  rep3: avg protein MW   33,182.3 g/mol |    100,921 molecules/cell | 3.012e+06 per um^3
  mean total proteins/cell: 100,457 +/- 330
  pseudogenes carry 0.02% of the iPM pool but have no MW, so they do not enter the average above


**Attach transcription.** Absolute mRNA copies per cell from
`Syn1_Syn3A_Transcriptomics/syn3A_rna_abundances.tsv` (written by
`Calc_Abundances_syn3A.py`).

In [10]:
rna = pd.read_csv(SYN3A_RNA, sep="\t").set_index("locus_tag")
syn3a["mRNA_copies_per_cell"] = syn3a["locus_tag"].map(rna["copies_per_cell"]).round(2)
n_rna = int(syn3a["mRNA_copies_per_cell"].notna().sum())
say(f"mRNA copies/cell from {SYN3A_RNA.name}: {n_rna} / {len(syn3a)}")

mRNA copies/cell from syn3A_rna_abundances.tsv: 458 / 458


**Validate the curation.** Every (Secondary, Tertiary) pair must exist in the
controlled vocabulary — this catches manual-edit slips in the curated workbook.

In [11]:
h = pd.read_csv(HIER, sep="\t")
h["Secondary"] = h["Secondary"].ffill()
legal = {(str(s).strip(), str(t).strip()) for s, t in zip(h["Secondary"], h["Tertiary"])
         if pd.notna(t) and str(t).strip()}

curated = syn3a[~syn3a["is_pseudogene"]]      # pseudogenes were never curated
bad = curated[[(str(s).strip(), str(t).strip()) not in legal
               for s, t in zip(curated["secondary_function"], curated["tertiary_function"])]]
say(f"\nlegal (Secondary, Tertiary) pairs in {HIER.name}: {len(legal)}")
say(f"  checked against the {len(curated)} curated proteins "
    f"({int(syn3a['is_pseudogene'].sum())} pseudogenes excluded, never curated)")
if bad.empty:
    say("  all emitted pairs are legal")
else:
    say(f"  ILLEGAL pairs: {len(bad)} protein(s)")
    for _, r in bad.iterrows():
        say(f"    {r['locus_tag']}  {str(r['gene_name']):<10} "
            f"{r['secondary_function']} / {r['tertiary_function']}")

say("\ncuration trail (Review Flag):")
for k, v in syn3a["review_flag"].value_counts(dropna=False).items():
    say(f"  {str(k):<28} {v:>4}")
say(f"  Mismatch Solved (adjudicated): {int(syn3a['mismatch_solved'].notna().sum())}")


legal (Secondary, Tertiary) pairs in function_hierachy.tsv: 70
  checked against the 455 curated proteins (3 pseudogenes excluded, never curated)
  ILLEGAL pairs: 3 protein(s)
    JCVISYN3A_0913  tetM       Exogenous / Exogenous
    JCVISYN3A_0918  hisB       Exogenous / Exogenous
    JCVISYN3A_0931  met14p     Exogenous / Exogenous

curation trail (Review Flag):
  AI                            224
  nan                           179
  PRIMARY_MISMATCH; AI           26
  PRIMARY_MISMATCH               23
  PSEUDOGENE_NOT_CURATED          3
  CONFLICT(also:Nucleotide)       2
  CONFLICT(also:Lipid)            1
  Mismatch Solved (adjudicated): 50


**Composition.** Gene-count share against measured copy-number share — a few classes dominate copy number on modest gene counts.

In [12]:
PRIM, SEC, TER = "primary_function", "secondary_function", "tertiary_function"
ABUND3 = "exp_ptn_cnt_2019"
N3 = len(syn3a)
total_copies3 = syn3a[ABUND3].sum()

def share_table(col, title):
    g = (syn3a.groupby(col, dropna=False)
              .agg(n_genes=("locus_tag", "size"), copies=(ABUND3, "sum")))
    g["pct_genes"] = 100 * g["n_genes"] / N3
    g["pct_copies"] = 100 * g["copies"] / total_copies3
    g = g.sort_values("copies", ascending=False)
    say(f"\n{title}")
    say(f"  {'category':<40} {'genes':>6} {'%gene':>7} {'copies':>10} {'%copy':>7}")
    for cat, r in g.iterrows():
        say(f"  {str(cat):<40} {int(r['n_genes']):>6} {r['pct_genes']:>6.1f}% "
            f"{int(r['copies']):>10} {r['pct_copies']:>6.1f}%")
    return g

say("")
say(f"proteins {N3} | total measured copies (2019): {int(total_copies3):,}")
say("\nEssentiality:")
for k, v in syn3a["essentiality"].value_counts(dropna=False).items():
    say(f"  {str(k):<24} {v:>4}  ({100*v/N3:.1f}%)")
say("\nLocalization:")
for k, v in syn3a["localization"].value_counts(dropna=False).items():
    say(f"  {str(k):<24} {v:>4}  ({100*v/N3:.1f}%)")
share_table(PRIM, "By Primary Function:")
share_table(SEC, "By Secondary Function:")

say("\nFunctional hierarchy (Primary > Secondary > Tertiary; n genes):")
grp = (syn3a.groupby([PRIM, SEC, TER], dropna=False)
            .agg(n_genes=("locus_tag", "size")).reset_index())
for prim in grp[PRIM].drop_duplicates():
    pblk = grp[grp[PRIM] == prim]
    say(f"\n  {prim}  (n={int(pblk['n_genes'].sum())})")
    for sec in pblk[SEC].drop_duplicates():
        sblk = pblk[pblk[SEC] == sec]
        say(f"    {sec}  (n={int(sblk['n_genes'].sum())})")
        for _, r in sblk.sort_values("n_genes", ascending=False).iterrows():
            say(f"        {str(r[TER]):<46} {int(r['n_genes']):>4}")


proteins 458 | total measured copies (2019): 76,687

Essentiality:
  Essential                 270  (59.0%)
  Quasiessential            113  (24.7%)
  Nonessential               72  (15.7%)
  nan                         3  (0.7%)

Localization:
  cytoplasm                 315  (68.8%)
  trans-membrane             84  (18.3%)
  peripheral membrane        38  (8.3%)
  lipoprotein                13  (2.8%)
  unidentified                3  (0.7%)
  nan                         3  (0.7%)
  extracellular               2  (0.4%)

By Primary Function:
  category                                  genes   %gene     copies   %copy
  Genetic Information Processing              215   46.9%      43839   57.2%
  Metabolism                                  154   33.6%      26174   34.1%
  Unclear                                      74   16.2%       5481    7.1%
  Cellular Processes                            8    1.7%        973    1.3%
  Environmental Information Processing          1    0.2%        

**Write** the syn3A table and its interactive page.

In [13]:
# The TSV is the pipeline contract: full precision, and it keeps iPM because nine
# downstream scripts normalise on it. The workbook and the page are the published
# views: no iPM, and protein counts rounded up to whole molecules.
SYN3A_COLS = ["locus_tag", "gene_name", "gene_product",
              "essentiality", "localization",
              "primary_function", "secondary_function", "tertiary_function",
              "exp_ptn_cnt_2019", "exp_ptn_cnt_2026", "sim_initial_ptn_cnt",
              "mRNA_copies_per_cell",
              "iPM_rep1", "iPM_rep2", "iPM_rep3", "iPM_mean", "iPM_CV",
              "protein_length", "protein_mw_Da", "protein_sequence",
              "review_flag", "mismatch_solved"]
SYN3A_PUB_COLS = [c for c in SYN3A_COLS
                  if not c.startswith("iPM_") and c not in ("review_flag", "mismatch_solved")]

syn3a_out = syn3a[SYN3A_COLS].copy()

def whole_copies(s):
    """Protein counts as whole molecules (rounded up); blanks stay blank."""
    return np.ceil(pd.to_numeric(s, errors="coerce")).astype("Int64")

syn3a_pub = syn3a_out[SYN3A_PUB_COLS].copy()
syn3a_pub["exp_ptn_cnt_2026"] = whole_copies(syn3a_pub["exp_ptn_cnt_2026"])
syn3a_pub["exp_ptn_cnt_2019"] = whole_copies(syn3a_pub["exp_ptn_cnt_2019"])
syn3a_out.to_csv(OUT_SYN3A_TSV, sep="\t", index=False)
say(f"\nwrote {OUT_SYN3A_TSV}  ({len(syn3a_out)} proteins, {len(SYN3A_COLS)} columns)")

# stable, print-friendly palette, shared with the comparison page and Genome_Reduction
PRIM_COLORS = {
    "Genetic Information Processing":       "#3b6db3",
    "Metabolism":                           "#3f9e5a",
    "Unclear":                              "#9aa0a6",
    "Cellular Processes":                   "#8e6bb1",
    "Environmental Information Processing": "#2aa6a0",
    "Exogenous":                            "#c0654e",
    "Pseudogene":                           "#b0a0c8",   # OUTPUT.md pseudogene colour
}
# cluster tertiary bars under their secondary only where there are enough of them
SECGROUP = {"Genetic Information Processing", "Metabolism"}

pc.write_proteome_html(
    OUT_SYN3A_HTML, syn3a_pub, SYN3A_PUB_COLS,
    [{"heading": "", "levels": [PRIM, SEC, TER], "cluster": True,
      "order": syn3a_pub[PRIM].value_counts().index.tolist()}],
    title=f"JCVI-syn3A proteome ({N3} proteins)",
    subtitle="Curated Primary &rsaquo; Secondary &rsaquo; Tertiary function hierarchy. ",
    palette=PRIM_COLORS, dl_basename="syn3A_proteins",
    seq_cols=["protein_sequence"],
    num_cols=["exp_ptn_cnt_2019", "exp_ptn_cnt_2026", "sim_initial_ptn_cnt",
              "mRNA_copies_per_cell", "protein_length", "protein_mw_Da"],
    sticky_widths=(140, 90, 320))
say(f"wrote {OUT_SYN3A_HTML}")
syn3a_pub.head()


wrote syn3A_proteome.tsv  (458 proteins, 22 columns)
wrote syn3A_proteome.html


,locus_tag,gene_name,gene_product,essentiality,localization,primary_function,secondary_function,tertiary_function,exp_ptn_cnt_2019,exp_ptn_cnt_2026,sim_initial_ptn_cnt,mRNA_copies_per_cell,protein_length,protein_mw_Da,protein_sequence
0,JCVISYN3A_0001,dnaA,Chromosomal replication initiator protein,Essential,cytoplasm,Genetic Information Processing,DNA Maintenance,DNA replication control,148,75,148.0,0.07,450.0,51702.7783,MNVNDILKELKLSLMANKNIDESVYNDYIKTINIHKKGFSDYIVVV...
1,JCVISYN3A_0002,dnaN,DNA polymerase III subunit beta,Essential,cytoplasm,Genetic Information Processing,DNA Maintenance,DNA replication complex,213,270,213.0,0.06,375.0,42649.6889,MNFSINRMVLLDNLSKAAKVIDPKNVNPSLAGIYLNVLSDQVNIIA...
2,JCVISYN3A_0003,rnmV,Ribonuclease M5,Quasiessential,cytoplasm,Genetic Information Processing,Translation,Ribosome biogenesis,65,49,65.0,0.02,180.0,20634.5673,MSKIKQIIIVEGKTDSDKLKSIYGNDLKTIQTKGLSLNKKTLEMIK...
3,JCVISYN3A_0004,ksgA,16S rRNA (adenine(1518)-N(6)/adenine(1519)-N(6...,Nonessential,cytoplasm,Genetic Information Processing,Translation,Ribosome biogenesis,40,18,40.0,0.02,266.0,31176.8505,MKAKKYYGQNFISDLNLINKIVDVLDQNKDQLIIEIGPGKGALTKE...
4,JCVISYN3A_0005,NaN,Uncharacterized protein,Nonessential,trans-membrane,Unclear,No Kegg ortholog,Function unknown,72,31,72.0,0.07,363.0,42663.1420,MIRDFNNQEVTLDDLEQNNNKTDKNKPKVQFLMRFSLVFSNISTHI...


---
# Part C — syn1 vs syn3A

Joined on locus number, which the two genomes share. syn1 genes retained in syn3A
inherit the curated function annotation; genes deleted during minimization have no
syn3A counterpart and therefore no annotation — they are labelled explicitly rather
than left blank.

Fold change is syn3A / syn1 on **iPM**, i.e. each protein's share of its own
organism's proteome. It is a *relative* measure: a protein can hold a larger share
of the smaller proteome without its absolute copy number rising. Copies per cell
are carried alongside for the absolute reading.

The heavier reallocation analysis — deletion-corrected pools, PTR, operon-level
effects — lives in `Genome_Reduction/09` and `10`. This page is a browsable join.

In [14]:
rule("Part C - syn1 vs syn3A")

s1 = syn1[["locus_num", "locus_tag", "gene_name", "gene_product", "rna_type",
           "ptn_copy_number", "ptn_localization"]].rename(
    columns={"locus_tag": "syn1_locus_tag",
             "ptn_copy_number": "syn1_exp_ptn_cnt_2026",
             "ptn_localization": "syn1_localization"})

s3 = syn3a[["locus_num", "locus_tag", "exp_ptn_cnt_2026", "localization",
            "essentiality", "is_pseudogene", PRIM, SEC, TER]].rename(
    columns={"locus_tag": "syn3A_locus_tag",
             "exp_ptn_cnt_2026": "syn3A_exp_ptn_cnt_2026",
             "localization": "syn3A_localization"})

cmp = s1.merge(s3, on="locus_num", how="outer", indicator=True)
say(f"syn1 loci {len(s1)} | syn3A proteins {len(s3)} | joined rows {len(cmp)}")
say("  " + " | ".join(f"{k}={v}" for k, v in cmp['_merge'].value_counts().items()))

# Status: what genome reduction did to each locus. A locus present in both is not
# necessarily unchanged -- 40 of them carry a different protein in syn3A (a length
# change, a point substitution, or a pseudogene/gene flip in either direction), so
# those are called out rather than lumped in with the untouched ones. syn1 protein
# sequences come from syn1_proteins.faa, verified identical to the syn1.gb CDS
# translations (max difference 0.0000 Da over all 828).
_faa1 = {h.split("|")[0]: v.rstrip("*")
         for h, v in pc.read_fasta(str(SYN1_FAA)).items()}
_seq1 = syn1.assign(_s=syn1["locus_tag"].map(_faa1)).drop_duplicates("locus_num") \
            .set_index("locus_num")["_s"].to_dict()
_seq3 = syn3a.drop_duplicates("locus_num").set_index("locus_num")["protein_sequence"].to_dict()

def classify(r):
    if pd.isna(r["syn3A_locus_tag"]):
        return "Deleted in Syn3A"
    if pd.isna(r["syn1_locus_tag"]):
        return "Absent from Syn1"          # the syn3A-only loci; nothing was retained
    a, b = _seq1.get(r["locus_num"]), _seq3.get(r["locus_num"])
    a = a if isinstance(a, str) and a else None
    b = b if isinstance(b, str) and b else None
    if a is None and b is None:
        return "Retained in Syn3A"         # no product on either side (RNA gene)
    if a is not None and b is not None:
        return "Retained in Syn3A" if a == b else "Changed annotation in Syn3A"
    return "Changed annotation in Syn3A"   # a product on one side only

cmp["status"] = cmp.apply(classify, axis=1)

def detection(r):
    a, b = pd.notna(r["syn1_exp_ptn_cnt_2026"]), pd.notna(r["syn3A_exp_ptn_cnt_2026"])
    if a and b:
        return "Detected in both"
    if a:
        return "Syn1 only"
    if b:
        return "Syn3A only"
    return "Not detected"

cmp["detection"] = cmp.apply(detection, axis=1)

# Fold change on ABSOLUTE copies per cell. Note the two organisms carry different
# total protein budgets (~127,800 vs ~100,400 molecules), so a ratio of 1 means the
# same number of molecules, not the same share of the proteome.
cmp["copies_fold_change"] = cmp["syn3A_exp_ptn_cnt_2026"] / cmp["syn1_exp_ptn_cnt_2026"]

# genes absent from syn3A have no curated function; say so rather than leave blanks
NOT_ANN = "Not annotated (absent from Syn3A)"
for c in (PRIM, SEC, TER):
    cmp[c] = cmp[c].fillna(NOT_ANN)

say("\nstatus:")
for k, v in cmp["status"].value_counts().items():
    say(f"  {k:<30} {v:>4}")
say("mass-spectrometry detection:")
for k, v in cmp["detection"].value_counts().items():
    say(f"  {k:<30} {v:>4}")

_ch = cmp[cmp["status"] == "Changed annotation in Syn3A"]
say(f"\nthe {len(_ch)} loci whose protein changed (syn1 aa -> syn3A aa):")
for _, r in _ch.iterrows():
    a, b = _seq1.get(r["locus_num"]), _seq3.get(r["locus_num"])
    la = len(a) if isinstance(a, str) and a else None
    lb = len(b) if isinstance(b, str) and b else None
    how = ("no product -> {} aa".format(lb) if la is None else
           "{} aa -> no product".format(la) if lb is None else
           "{} aa, substitution".format(la) if la == lb else
           "{} -> {} aa".format(la, lb))
    say(f"    {r['syn1_locus_tag']} / {r['syn3A_locus_tag']}  "
        f"{str(r['gene_name'] or ''):<9} {how}")

both = cmp[cmp["detection"] == "Detected in both"]
say(f"\ncopy-number fold change (syn3A/syn1) over {len(both)} proteins detected in both:")
say(f"  median {both['copies_fold_change'].median():.3f} | "
    f"Pearson r on log10 copies = "
    f"{np.corrcoef(np.log10(both['syn1_exp_ptn_cnt_2026']), np.log10(both['syn3A_exp_ptn_cnt_2026']))[0,1]:.3f}")


Part C - syn1 vs syn3A
syn1 loci 911 | syn3A proteins 458 | joined rows 914
  left_only=456 | both=455 | right_only=3

status:
  Deleted in Syn3A                456
  Retained in Syn3A               415
  Changed annotation in Syn3A      40
  Absent from Syn1                  3
mass-spectrometry detection:
  Detected in both                423
  Syn1 only                       298
  Not detected                    167
  Syn3A only                       26

the 40 loci whose protein changed (syn1 aa -> syn3A aa):
    MMSYN1_0034 / JCVISYN3A_0034            1789 -> 1795 aa
    MMSYN1_0051 / JCVISYN3A_0051            98 aa -> no product
    MMSYN1_0113 / JCVISYN3A_0113            414 -> 449 aa
    MMSYN1_0213 / JCVISYN3A_0213  eno       451 aa, substitution
    MMSYN1_0235 / JCVISYN3A_0235            96 -> 60 aa
    MMSYN1_0262 / JCVISYN3A_0262  rpe       no product -> 225 aa
    MMSYN1_0286 / JCVISYN3A_0286            no product -> 268 aa
    MMSYN1_0325 / JCVISYN3A_0325            496 

**Pseudogenes.** syn1 carries 42 features NCBI marks pseudo; how many survived minimization.

In [15]:
pseudo = cmp[cmp["rna_type"] == "pseudo"]
say(f"\nsyn1 pseudogenes: {len(pseudo)}")
for k, v in pseudo["status"].value_counts().items():
    say(f"  {k:<30} {v:>4}")
# "Changed annotation" here means syn3A calls a protein where syn1 called a
# pseudogene -- the interesting direction, so keep those in the listing
kept = pseudo[pseudo["status"] != "Deleted in Syn3A"]
if len(kept):
    say("  retained, with syn3A copies/cell:")
    for _, r in kept.iterrows():
        say(f"    {r['syn1_locus_tag']}  {str(r['gene_name'] or ''):<8} "
            f"{str(r['gene_product'])[:44]:<44} "
            f"{'' if pd.isna(r['syn3A_exp_ptn_cnt_2026']) else f'{r.syn3A_exp_ptn_cnt_2026:8.1f}'}")

say(f"\nsyn3A pseudogenes carried in the proteome: "
    f"{int(cmp['is_pseudogene'].fillna(False).sum())}")
for _, r in cmp[cmp["is_pseudogene"].fillna(False)].iterrows():
    say(f"    {r['syn3A_locus_tag']}  {str(r['gene_product'])[:52]:<52} "
        f"{r['syn3A_exp_ptn_cnt_2026']:9.1f} copies/cell")


syn1 pseudogenes: 42
  Deleted in Syn3A                 33
  Changed annotation in Syn3A       7
  Retained in Syn3A                 2
  retained, with syn3A copies/cell:
    MMSYN1_0262  rpe      PSEUDOGENE -ribulose-phosphate 3-epimerase      117.9
    MMSYN1_0286           PSEUDOGENE - conserved hypothetical protein       3.7
    MMSYN1_0353           PSEUDOGENE - DivIVA domain protein              226.9
    MMSYN1_0399           PSEUDOGENE - efflux ABC transporter_ permeas     17.5
    MMSYN1_0401           PSEUDOGENE - peptidase C39 family protein         0.2
    MMSYN1_0479           PSEUDOGENE - conserved hypothetical protein      32.0
    MMSYN1_0546           PSEUDOGENE - putative hydrolase of the HAD f      3.4
    MMSYN1_0602           PSEUDOGENE - conserved hypothetical protein      16.1
    MMSYN1_0887           PSEUDOGENE - pyridine nucleotide-disulphide      26.2

syn3A pseudogenes carried in the proteome: 3
    JCVISYN3A_0051  conserved hypothetical protein            

/tmp/ipykernel_2255788/1889078273.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  f"{int(cmp['is_pseudogene'].fillna(False).sum())}")
/tmp/ipykernel_2255788/1889078273.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  for _, r in cmp[cmp["is_pseudogene"].fillna(False)].iterrows():


**Write** the comparison page.

In [16]:
CMP_COLS = ["locus_num", "gene_name", "gene_product", "status",
            "syn1_locus_tag", "syn3A_locus_tag",
            "syn1_exp_ptn_cnt_2026", "syn3A_exp_ptn_cnt_2026", "copies_fold_change",
            PRIM, SEC, TER, "essentiality",
            "syn1_localization", "syn3A_localization", "rna_type"]

cmp_out = cmp[CMP_COLS].copy().sort_values("locus_num").reset_index(drop=True)
cmp_out["copies_fold_change"] = cmp_out["copies_fold_change"].astype(float).round(3)
for c in ["syn1_exp_ptn_cnt_2026", "syn3A_exp_ptn_cnt_2026"]:
    cmp_out[c] = whole_copies(cmp_out[c])

STATUS_COLORS = {"Retained in Syn3A": "#3182bd", "Deleted in Syn3A": "#c0392b",
                 "Changed annotation in Syn3A": "#d98b30", "Absent from Syn1": "#7b8794"}
PRIM_COLORS = {**PRIM_COLORS, "Pseudogene": "#b0a0c8"}
PALETTE = {**PRIM_COLORS, **STATUS_COLORS, NOT_ANN: "#c0392b"}

pc.write_proteome_html(
    OUT_CMP_HTML, cmp_out, CMP_COLS,
    [{"heading": "", "levels": [PRIM, SEC, TER], "cluster": True, "balance": True}],
    title=f"JCVI-syn1.0 vs JCVI-syn3A proteome ({len(cmp_out)} loci)",
    subtitle="Joined on locus number. Fold change is syn3A/syn1 on absolute copies "
             "per cell &mdash; the two cells carry different total protein budgets "
             "(~127,800 vs ~100,400 molecules), so a ratio of 1 means the same number "
             "of molecules, not the same share of the proteome. ",
    palette=PALETTE, dl_basename="syn1_vs_syn3A_proteins",
    num_cols=["syn1_exp_ptn_cnt_2026", "syn3A_exp_ptn_cnt_2026", "copies_fold_change"],
    sticky_widths=(90, 90, 300))
say(f"\nwrote {OUT_CMP_HTML}  ({len(cmp_out)} loci)")
cmp_out.head()


wrote syn1_vs_syn3A_proteome.html  (914 loci)


,locus_num,gene_name,gene_product,status,syn1_locus_tag,syn3A_locus_tag,syn1_exp_ptn_cnt_2026,syn3A_exp_ptn_cnt_2026,copies_fold_change,primary_function,secondary_function,tertiary_function,essentiality,syn1_localization,syn3A_localization,rna_type
0,0001,dnaA_1,chromosomal replication initiator protein DnaA,Retained in Syn3A,MMSYN1_0001,JCVISYN3A_0001,29,75,2.581,Genetic Information Processing,DNA Maintenance,DNA replication control,Essential,cytoplasmic,cytoplasm,mRNA
1,0002,dnaN,DNA polymerase III_ beta subunit,Retained in Syn3A,MMSYN1_0002,JCVISYN3A_0002,207,270,1.302,Genetic Information Processing,DNA Maintenance,DNA replication complex,Essential,cytoplasmic,cytoplasm,mRNA
2,0003,rnmV,ribonuclease M5,Retained in Syn3A,MMSYN1_0003,JCVISYN3A_0003,39,49,1.269,Genetic Information Processing,Translation,Ribosome biogenesis,Quasiessential,cytoplasmic,cytoplasm,mRNA
3,0004,ksgA,dimethyladenosine transferase,Retained in Syn3A,MMSYN1_0004,JCVISYN3A_0004,29,18,0.589,Genetic Information Processing,Translation,Ribosome biogenesis,Nonessential,cytoplasmic,cytoplasm,mRNA
4,0005,,hypothetical purine NTPase,Retained in Syn3A,MMSYN1_0005,JCVISYN3A_0005,53,31,0.580,Unclear,No Kegg ortholog,Function unknown,Nonessential,membrane,trans-membrane,mRNA


---
# Part D — distribution workbooks and log

One workbook per organism, each with the proteome on a `Proteome` sheet and a
`Columns` sheet defining every field and its units, so the table travels without
this notebook.

In [17]:
SYN1_LEGEND = {
    "locus_tag": "JCVI-syn1.0 locus tag (CP002027.1)",
    "gene_name": "gene symbol where assigned",
    "gene_product": "product description from the GFF3 annotation",
    "rna_type": "mRNA | tRNA | rRNA | ncRNA | pseudo",
    "chrom": "reference sequence accession",
    "start0": "gene start, 0-based inclusive",
    "end0": "gene end, 0-based exclusive",
    "strand": "+ or -",
    "protein_mw_Da": "protein molecular weight (Da), from the GenBank CDS translation",
    "protein_length": "protein length (aa), from DeepTMHMM",
    "ptn_copy_number": "estimated protein copies per cell (whole molecules, rounded up)",
    "TMRs": "number of transmembrane regions predicted by DeepTMHMM",
    "signalP": "SignalP 6 call: extracellular | lipoprotein | blank",
    "ptn_localization": ("cytoplasmic | membrane | extracellular | lipoprotein | "
                         "Non_proteins (RNA gene) | Non_predicted"),
}
SYN3A_LEGEND = {
    "locus_tag": "JCVI-syn3A locus tag (CP016816.2)",
    "gene_name": "gene symbol where assigned",
    "gene_product": "product description",
    "essentiality": "syn3A design essentiality class",
    "localization": "curated subcellular localization",
    "primary_function": "curated function, level 1",
    "secondary_function": "curated function, level 2",
    "tertiary_function": "curated function, level 3",
    "exp_ptn_cnt_2019": "measured protein copies per cell, 2019 dataset (whole molecules)",
    "exp_ptn_cnt_2026": "estimated protein copies per cell, this study (whole molecules, rounded up)",
    "sim_initial_ptn_cnt": "initial protein count used by the whole-cell model",
    "mRNA_copies_per_cell": "absolute mRNA copies per cell (Calc_Abundances_syn3A.py)",
    "protein_length": "protein length (aa); blank for pseudogenes",
    "protein_mw_Da": ("protein molecular weight (Da), computed from the sequence; "
                      "blank for pseudogenes"),
    "protein_sequence": "amino-acid sequence (stop codon removed); blank for pseudogenes",
}

# ── Membrane topology ────────────────────────────────────────────────────────
# The Proteome sheet keeps only the helix count and the final localization call.
# This sheet carries DeepTMHMM's full segment layout -- where each inside /
# TMhelix / outside / signal run starts and ends -- which is what you need to
# place a protein in the membrane rather than just classify it.
def topology_record(tag):
    t = topo.get(tag)
    if not t or not t["regions"]:
        return None
    segs = t["regions"]
    return {
        "protein_length": t["length"],
        "n_TM_helices": t["TMRs"],
        "n_segments": len(segs),
        "signal_peptide_region": next((f"{a}-{b}" for k, a, b in segs if k == "signal"), ""),
        "TM_segments": ", ".join(f"{a}-{b}" for k, a, b in segs if k == "TMhelix"),
        "topology": " | ".join(f"{k}:{a}-{b}" for k, a, b in segs),
    }

TOPO_BY_NUM = {}
for _t, _n, _sp in zip(syn1["locus_tag"], syn1["locus_num"], syn1["signalP"]):
    _r = topology_record(_t)
    if _r:
        TOPO_BY_NUM[_n] = dict(_r, syn1_locus_tag=_t, signal_peptide=_sp)

TOPO_COLS = ["locus_tag", "gene_name", "protein_length", "n_TM_helices", "n_segments",
             "signal_peptide", "signal_peptide_region", "TM_segments", "topology",
             "localization"]

def topology_sheet(df, loc_col, keep_source=False):
    """One row per protein that DeepTMHMM predicted, in locus order."""
    rows = []
    for _, r in df.iterrows():
        rec = TOPO_BY_NUM.get(r["locus_num"])
        if rec is None:
            continue
        # SignalP, like the topology, was predicted on the syn1 protein
        row = {"locus_tag": r["locus_tag"], "gene_name": r.get("gene_name"),
               "localization": r[loc_col],
               **{k: v for k, v in rec.items() if k != "syn1_locus_tag"}}
        if keep_source:
            row["predicted_on"] = rec["syn1_locus_tag"]
        rows.append(row)
    cols = TOPO_COLS + (["predicted_on"] if keep_source else [])
    return pd.DataFrame(rows)[cols]


TOPO_LEGEND = {
    "locus_tag": "locus tag",
    "gene_name": "gene symbol where assigned",
    "protein_length": "protein length (aa) as seen by DeepTMHMM",
    "n_TM_helices": "number of predicted transmembrane helices",
    "n_segments": "number of topology segments the protein is divided into",
    "signal_peptide": "SignalP 6 call: extracellular | lipoprotein | blank",
    "signal_peptide_region": "residue range of the signal peptide, if any",
    "TM_segments": "residue ranges of the transmembrane helices, N- to C-terminal",
    "topology": ("full segment layout N- to C-terminal, as "
                 "<segment>:<first>-<last>; inside = cytoplasmic face, "
                 "outside = extracellular face"),
    "localization": "final localization assignment carried on the Proteome sheet",
    "predicted_on": "syn1 locus the prediction was made on (same locus number)",
}


def write_workbook(path, df, legend, topo_df=None, sheet="Proteome"):
    """Proteome sheet, optional Topology sheet, and a Columns legend for both."""
    parts = [("Proteome", df, legend)]
    if topo_df is not None:
        parts.append(("Topology", topo_df, TOPO_LEGEND))
    leg = pd.DataFrame(
        [{"Sheet": name, "Column": c, "Description": lg.get(c, "")}
         for name, frame, lg in parts for c in frame.columns])
    with pd.ExcelWriter(path, engine="openpyxl") as xl:
        df.to_excel(xl, sheet_name=sheet, index=False)
        if topo_df is not None:
            topo_df.to_excel(xl, sheet_name="Topology", index=False)
        leg.to_excel(xl, sheet_name="Columns", index=False)
    return path

rule("Part D - distribution workbooks")
syn1_pub = syn1_out[SYN1_PUB_COLS].copy()
syn1_pub["ptn_copy_number"] = whole_copies(syn1_pub["ptn_copy_number"])

# syn1 was predicted directly; syn3A inherits by locus number, the two genomes
# sharing both their numbering and (for retained genes) their protein sequences.
topo1 = topology_sheet(syn1, loc_col="ptn_localization")
topo3 = topology_sheet(syn3a, loc_col="localization", keep_source=True)

write_workbook(OUT_SYN1_XLSX, syn1_pub, SYN1_LEGEND, topo_df=topo1)
say(f"wrote {OUT_SYN1_XLSX}  ({len(syn1_pub)} loci x {len(syn1_pub.columns)} cols, "
    f"Topology {len(topo1)} proteins, Columns legend)")
write_workbook(OUT_SYN3A_XLSX, syn3a_pub, SYN3A_LEGEND, topo_df=topo3)
say(f"wrote {OUT_SYN3A_XLSX}  ({len(syn3a_pub)} proteins x {len(syn3a_pub.columns)} cols, "
    f"Topology {len(topo3)} proteins, Columns legend)")
say(f"  topology: {int((topo1['n_TM_helices'] > 0).sum())} syn1 and "
    f"{int((topo3['n_TM_helices'] > 0).sum())} syn3A proteins carry at least one TM helix")

undoc1 = [c for c in syn1_pub.columns if c not in SYN1_LEGEND]
undoc3 = [c for c in syn3a_pub.columns if c not in SYN3A_LEGEND]
assert not undoc1 and not undoc3, f"undocumented columns: {undoc1} {undoc3}"
say("every column carries a description")


Part D - distribution workbooks


wrote syn1_proteome_2026.xlsx  (911 loci x 14 cols, Topology 828 proteins, Columns legend)
wrote Syn3A_proteome_2026.xlsx  (458 proteins x 15 cols, Topology 446 proteins, Columns legend)
  topology: 162 syn1 and 83 syn3A proteins carry at least one TM helix
every column carries a description


In [18]:
rule("Summary")
say(f"syn1  : {len(syn1_out):>4} loci, {n_det} with mass-spec signal, "
    f"{total1.mean():,.0f} protein molecules/cell")
say(f"syn3A : {len(syn3a_out):>4} proteins ({int(syn3a['is_pseudogene'].sum())} pseudogenes), "
    f"{int(syn3a_out['iPM_mean'].notna().sum())} with mass-spec signal, "
    f"{total3.mean():,.0f} protein molecules/cell")
say(f"joined: {len(cmp_out):>4} loci | "
    f"{int((cmp_out['status'] == 'Retained in Syn3A').sum())} retained, "
    f"{int((cmp_out['status'] == 'Deleted in Syn3A').sum())} deleted in syn3A")
say("")
for p in [OUT_SYN1_TSV, OUT_SYN3A_TSV, OUT_SYN1_XLSX, OUT_SYN3A_XLSX,
          OUT_SYN3A_HTML, OUT_CMP_HTML]:
    say(f"  {p}  ({p.stat().st_size/1024:,.0f} kB)")

OUT_LOG.write_text("\n".join(_LOG) + "\n")
print(f"\nwrote {OUT_LOG}")


Summary
syn1  :  911 loci, 721 with mass-spec signal, 127,847 protein molecules/cell
syn3A :  458 proteins (3 pseudogenes), 449 with mass-spec signal, 100,457 protein molecules/cell
joined:  914 loci | 415 retained, 456 deleted in syn3A

  syn1_proteome.tsv  (186 kB)
  syn3A_proteome.tsv  (283 kB)
  syn1_proteome_2026.xlsx  (134 kB)
  Syn3A_proteome_2026.xlsx  (185 kB)
  syn3A_proteome.html  (409 kB)
  syn1_vs_syn3A_proteome.html  (524 kB)

wrote Proteome.txt
